### Embedding model form scrath for similarity search using the cosine similariy

Here is the full pipeline steps that we are going to follow,

- Text processing - Here we need to process the dataset from lowercaseing, strip punctuations and also remove noice.
- Tokenization - then we need to create the token ids from the text for this we can use the BPE
- Vocb and Enbedding layer -  Here we can create the vocab from the data and create the embeddings
- Encoder layer (Transformer layer) - A small transformer or MLP that we can add the contextual meaning.
- L2 Norm - From this we are going to project all vectors onto the unit hypersphere and this will make sure the cosine similarity will be equivalent to dot-product.
- Triplet Loss - This will help to get the similar tokens closer and apart the dissimilar ones.

### Step 01

Here we will load the dataset that we are gonna use, we are using the triplet sunset of the data where that contains,
- anchor
- positive
- negative

### The difference of an Embedding model and a Tokenizer

#### Tokenizer - Here a tokenizer will take a sentence and create token ids from that, see the example.

```
text :"Deep learning models are cool"

then this will be tokenized as
["Deep", " learning", " models", " are", " cool"]

then this will asign a tokenid
[5021, 1821, 993, 45, 812]

```

In [9]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch , torch.nn as nn , torch.nn.functional as F

# load the dataset
dataset  =  load_dataset("sentence-transformers/all-nli", "triplet", split="train")

In [10]:
dataset[0]

{'anchor': 'A person on a horse jumps over a broken down airplane.',
 'positive': 'A person is outdoors, on a horse.',
 'negative': 'A person is at a diner, ordering an omelette.'}

In [11]:
tokenizer  =  AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [12]:
# Test the tokenizer
tokenizer(dataset[0]['anchor'], padding=True, truncation=True, return_tensors="pt")


{'input_ids': tensor([[   32,  1048,   319,   257,  8223, 18045,   625,   257,  5445,   866,
         19401,    13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

### Class for the Triplet Dataset

In [13]:
class  TripletDataset(torch.utils.data.Dataset):

    def __init__(self , data , tokenizer , max_length=32):
        super().__init__()
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self ):
        return len(self.data)
    
    def encode(self , text):
        return self.tokenizer(text , padding="max_length" , truncation=True , max_length=self.max_length , return_tensors="pt")["input_ids"].squeeze(0)
    
    def __getitem__(self , idx):
        anchor = self.data[idx]['anchor']
        positive = self.data[idx]['positive']
        negative = self.data[idx]['negative']

        return {
            "anchor": self.encode(anchor),
            "positive": self.encode(positive),
            "negative": self.encode(negative)
        }

In [14]:
# use the TripletDataset class
triplet_dataset = TripletDataset(dataset , tokenizer)
triplet_dataset[0]

{'anchor': tensor([   32,  1048,   319,   257,  8223, 18045,   625,   257,  5445,   866,
         19401,    13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256]),
 'positive': tensor([   32,  1048,   318, 24349,    11,   319,   257,  8223,    13, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256]),
 'negative': tensor([   32,  1048,   318,   379,   257, 47519,    11, 16216,   281,   267,
          1326, 21348,    13, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256])}

### Model Architecture

Here are the different layers that are in the model architecture.

1. Embedding layer - When a token ID comes in, you just retrieve its row. That's it mechanically, but what makes it powerful is that these vectors are not fixed. They start random and get updated during training via backpropagation.

2. Transformer Encoder - The embedding layer gives each token an independent vector with no awareness of the other tokens around it. The Transformer encoder is what introduces context. It applies self-attention, which lets every token look at every other token in the sequence and decide how much to borrow from each one.

3. Mean Pooling -Mean pooling simply averages all the token vectors together into a single vector. This is deliberately simple. More complex options exist (like using a special [CLS] token the way BERT does) but mean pooling consistently performs well for short text and requires no additional learned parameters.

3. Projection - After pooling we apply one linear layer that maps from the encoder's dimension (256) down to the final output dimension (128). This serves two purposes. 

    -  First, it gives the model a learned bottleneck — a final compression step that forces the most important semantic signal to survive while discarding noise. 
 
    - Second, it decouples the internal working dimension from the output dimension, so you can tune them independently. A 128-dimensional output vector is fast to compare and store, while the 256-dimensional internal space gives the encoder room to work.

4. L2 Norm - The very last operation divides every output vector by its own magnitude, so every embedding lands on the surface of a unit hypersphere.

    -  cosine similarity between two normalized vectors equals their dot product. That means similarity search becomes a single matrix multiply with no division, which is dramatically faster at scale and compatible with optimized libraries like FAISS.